In [ ]:
%pip install -U "torchao>=0.16.0"

In [ ]:
import torch
import peft
import diffusers
import torchao

print("torch:", torch.__version__)
print("peft:", peft.__version__)
print("diffusers:", diffusers.__version__)
print("torchao:", torchao.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)

pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/yamazaki-shishi-sdxl-dataset/images")

images = [
    p for p in INPUT_DIR.rglob("*")
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
]

print("画像枚数:", len(images))
for p in images[:10]:
    print(p.name)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("input exists:", root.exists())

for p in sorted(root.iterdir()):
    print("DIR:", p)

    if p.is_dir():
        for child in sorted(p.iterdir())[:20]:
            print("   ", child)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

images = [
    p for p in root.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
]

print("画像枚数:", len(images))

for p in images[:20]:
    print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("データセット一覧:")
for p in root.iterdir():
    print(p.name, "DIR" if p.is_dir() else "FILE")

print("\n画像ファイル検索:")
extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

images = [
    p for p in root.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
]

print("画像枚数:", len(images))
for p in images[:20]:
    print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

for p in root.iterdir():
    print(p)

images = [
    p for p in root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
]

print("画像枚数:", len(images))
for p in images[:10]:
    print(p)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("Input exists:", root.exists())
print("Input entries:", list(root.iterdir()))

target = root / "yamazaki-shishi-sdxl-dataset"
print("Expected dataset exists:", target.exists())

if target.exists():
    for p in list(target.rglob("*"))[:30]:
        print(p)

In [ ]:

from pathlib import Path

root = Path("/kaggle/input")

for p in root.rglob("*"):
    if p.is_dir():
        print("DIR ", p)
    elif p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
        print("IMAGE", p)


In [ ]:
INPUT_DIR = "/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset/images"

In [ ]:

from pathlib import Path
from PIL import Image

INPUT_DIR = Path(
    "/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset/images"
)

images = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)

print("画像枚数:", len(images))
print("サンプル:", [p.name for p in images[:5]])

if images:
    with Image.open(images[0]) as img:
        print("画像サイズ:", img.size)
        display(img.copy())


In [ ]:

import os
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

from diffusers import StableDiffusionXLPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

# -------------------------
# Settings
# -------------------------
INPUT_DIR = Path(
    "/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset/images"
)
OUTPUT_DIR = Path("/kaggle/working/yamazaki_shishi_lora")
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

TRIGGER = "yamazaki_shishi"
RES_W, RES_H = 768, 432
TRAIN_STEPS = 300
LEARNING_RATE = 1e-4
SEED = 42

assert torch.cuda.is_available(), "KaggleのGPUを有効にしてください"
assert INPUT_DIR.is_dir(), f"画像フォルダがありません: {INPUT_DIR}"

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

image_paths = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
assert image_paths, "学習画像が0枚です"

print("GPU:", torch.cuda.get_device_name(0))
print("Images:", len(image_paths))

# -------------------------
# Dataset
# -------------------------
CAPTION = (
    "yamazaki_shishi, Japanese traditional shishimai lion dance, "
    "traditional Japanese lion costume"
)

class ShishiDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as im:
            im = im.convert("RGB").resize(
                (RES_W, RES_H), Image.Resampling.LANCZOS
            )
            arr = np.asarray(im, dtype=np.float32) / 127.5 - 1.0

        pixels = torch.from_numpy(arr).permute(2, 0, 1)
        return {"pixel_values": pixels, "caption": CAPTION}

loader = DataLoader(
    ShishiDataset(image_paths),
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

# -------------------------
# Load SDXL
# -------------------------
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")

noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# Freeze original model weights
pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)
pipe.text_encoder_2.requires_grad_(False)
pipe.unet.requires_grad_(False)

# Inject trainable LoRA layers
lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    lora_dropout=0.0,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)

pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

pipe.unet.enable_gradient_checkpointing()
pipe.unet.train()

optimizer = torch.optim.AdamW(
    [p for p in pipe.unet.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

# Cache text conditioning for the fixed caption
with torch.no_grad():
    encoded = pipe.encode_prompt(
        prompt=CAPTION,
        device="cuda",
        num_images_per_prompt=1,
        do_classifier_free_guidance=False,
    )
    prompt_embeds = encoded[0].detach()
    pooled_prompt_embeds = encoded[2].detach()

prompt_embeds = prompt_embeds.to(dtype=torch.float16)
pooled_prompt_embeds = pooled_prompt_embeds.to(dtype=torch.float16)

# SDXL time IDs: original size, crop coordinates, target size
time_ids = torch.tensor(
    [[720, 1280, 0, 0, RES_H, RES_W]],
    device="cuda",
    dtype=torch.float16,
)

# -------------------------
# Training loop
# -------------------------
global_step = 0
loss_sum = 0.0

while global_step < TRAIN_STEPS:
    for batch in loader:
        if global_step >= TRAIN_STEPS:
            break

        pixel_values = batch["pixel_values"].to(
            device="cuda", dtype=torch.float16
        )

        # Encode image into latent space
        with torch.no_grad():
            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device="cuda",
        ).long()

        noisy_latents = noise_scheduler.add_noise(
            latents, noise, timesteps
        )

        # Predict noise; only LoRA parameters are trainable
        with torch.autocast("cuda", dtype=torch.float16):
            prediction = pipe.unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=prompt_embeds,
                added_cond_kwargs={
                    "text_embeds": pooled_prompt_embeds,
                    "time_ids": time_ids,
                },
            ).sample

            target = noise
            if noise_scheduler.config.prediction_type == "v_prediction":
                target = noise_scheduler.get_velocity(
                    latents, noise, timesteps
                )

            loss = F.mse_loss(
                prediction.float(), target.float(), reduction="mean"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in pipe.unet.parameters() if p.requires_grad],
            1.0,
        )
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        global_step += 1
        loss_sum += loss.item()

        if global_step % 25 == 0:
            print(
                f"Step {global_step}/{TRAIN_STEPS} "
                f"loss={loss_sum / 25:.5f}"
            )
            loss_sum = 0.0

# -------------------------
# Save standalone Diffusers LoRA
# -------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

peft_state = get_peft_model_state_dict(pipe.unet)
diffusers_state = convert_state_dict_to_diffusers(peft_state)

StableDiffusionXLPipeline.save_lora_weights(
    save_directory=str(OUTPUT_DIR),
    unet_lora_layers=diffusers_state,
    safe_serialization=True,
)

print("LoRA saved:", OUTPUT_DIR)
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])

# Release GPU memory
del optimizer, pipe
torch.cuda.empty_cache()


In [1]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("VAE dtype:", next(pipe.vae.parameters()).dtype)
print("UNet dtype:", next(pipe.unet.parameters()).dtype)

print("VAE scaling factor:", pipe.vae.config.scaling_factor)
print("Scheduler prediction type:", noise_scheduler.config.prediction_type)

GPU: Tesla T4
CUDA: 12.8


NameError: name 'pipe' is not defined

In [2]:
import torch
from pathlib import Path

INPUT_DIR = Path(
    "/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset/images"
)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("CUDA:", torch.version.cuda)
print("画像枚数:", len(list(INPUT_DIR.glob("*.png"))))
print("pipe exists:", "pipe" in globals())

GPU: Tesla T4
CUDA: 12.8
画像枚数: 144
pipe exists: False


In [ ]:
pipe.vae.to(dtype=torch.float32)

with torch.no_grad():
    pixels_fp32 = pixel_values.float()
    latents = pipe.vae.encode(
        pixels_fp32
    ).latent_dist.sample()

    latents = latents * pipe.vae.config.scaling_factor

    if not torch.isfinite(latents).all():
        raise RuntimeError("VAE latents contain NaN or Inf")

    latents = latents.to(dtype=torch.float16)

In [ ]:
# Kaggle Notebook script: make-lora.py
# Dataset input: /kaggle/input/yamazaki-shishi-sdxl-dataset
# Base model: stabilityai/stable-diffusion-xl-base-1.0
# Trigger token: yamazaki_shishi

import os
import torch
from diffusers import StableDiffusionXLPipeline
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from PIL import Image

# Paths
INPUT_DIR = "/kaggle/input/yamazaki-shishi-sdxl-dataset/images"
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load base SDXL
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
).to("cuda")

# Simple LoRA config (example using PEFT)
lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

# Training loop placeholder
# Actual training uses Kohya_ss / AI-Toolkit / ComfyUI LoRA node
# Save base weights for reference
pipe.save_pretrained(os.path.join(OUTPUT_DIR, "lora-yamazaki-shishi"))
print("LoRA dataset prep complete. Use Kohya_ss / AI-Toolkit for full training.")


In [3]:

import os
import gc
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

from diffusers import StableDiffusionXLPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

# Configuration
INPUT_DIR = Path(
    "/kaggle/input/datasets/tyokouchi/yamazaki-shishi-sdxl-dataset/images"
)
OUTPUT_DIR = Path("/kaggle/working/yamazaki_shishi_lora")
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

TRIGGER = "yamazaki_shishi"
WIDTH, HEIGHT = 512, 288
TRAIN_STEPS = 300
LEARNING_RATE = 1e-5
SEED = 42

assert torch.cuda.is_available(), "GPUを有効にしてください"
assert INPUT_DIR.is_dir(), f"画像フォルダがありません: {INPUT_DIR}"

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

image_paths = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
)
assert image_paths, "学習画像がありません"

print("GPU:", torch.cuda.get_device_name(0))
print("Images:", len(image_paths))

# Dataset
CAPTION = (
    "yamazaki_shishi, Japanese traditional shishimai lion dance, "
    "traditional Japanese lion costume"
)

class ShishiDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB").resize(
                (WIDTH, HEIGHT), Image.Resampling.LANCZOS
            )
            array = np.asarray(image, dtype=np.float32) / 127.5 - 1.0

        return {
            "pixel_values": torch.from_numpy(array).permute(2, 0, 1)
        }

loader = DataLoader(
    ShishiDataset(image_paths),
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

# Load model
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")

scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# Keep VAE in FP32 for encoding stability
pipe.vae.requires_grad_(False)
pipe.vae.to(dtype=torch.float32)

pipe.text_encoder.requires_grad_(False)
pipe.text_encoder_2.requires_grad_(False)
pipe.unet.requires_grad_(False)

# Attach LoRA
config = LoraConfig(
    r=4,
    lora_alpha=4,
    lora_dropout=0.0,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
pipe.unet = get_peft_model(pipe.unet, config)

# Keep trainable LoRA parameters in FP32
for parameter in pipe.unet.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

pipe.unet.enable_gradient_checkpointing()
pipe.unet.train()
pipe.unet.print_trainable_parameters()

trainable = [p for p in pipe.unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)

# Encode the fixed caption once
with torch.no_grad():
    result = pipe.encode_prompt(
        prompt=CAPTION,
        device="cuda",
        num_images_per_prompt=1,
        do_classifier_free_guidance=False,
    )
    prompt_embeds = result[0].detach().to(torch.float16)
    pooled_embeds = result[2].detach().to(torch.float16)

time_ids = torch.tensor(
    [[HEIGHT, WIDTH, 0, 0, HEIGHT, WIDTH]],
    device="cuda",
    dtype=torch.float16,
)

# Training
step = 0
running_loss = 0.0

while step < TRAIN_STEPS:
    for batch in loader:
        if step >= TRAIN_STEPS:
            break

        pixels = batch["pixel_values"].to("cuda", dtype=torch.float32)

        # VAE encoding in FP32
        with torch.no_grad():
            latents = pipe.vae.encode(pixels).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor

        if not torch.isfinite(latents).all():
            raise RuntimeError("NaN/Inf detected in VAE latents")

        latents = latents.to(torch.float16)
        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            0,
            scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device="cuda",
        ).long()

        noisy_latents = scheduler.add_noise(latents, noise, timesteps)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            prediction = pipe.unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=prompt_embeds,
                added_cond_kwargs={
                    "text_embeds": pooled_embeds,
                    "time_ids": time_ids,
                },
            ).sample

            target = noise
            if scheduler.config.prediction_type == "v_prediction":
                target = scheduler.get_velocity(latents, noise, timesteps)

            loss = F.mse_loss(
                prediction.float(), target.float(), reduction="mean"
            )

        if not torch.isfinite(prediction).all():
            raise RuntimeError(
                f"NaN/Inf in UNet prediction at step {step + 1}"
            )
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"NaN/Inf loss at step {step + 1}: {loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(trainable, 1.0)

        # Catch invalid gradients before changing weights
        if any(
            p.grad is not None and not torch.isfinite(p.grad).all()
            for p in trainable
        ):
            raise RuntimeError(f"NaN/Inf gradient at step {step + 1}")

        optimizer.step()

        step += 1
        running_loss += loss.item()

        if step <= 5 or step % 25 == 0:
            print(
                f"Step {step}/{TRAIN_STEPS} | "
                f"loss={loss.item():.6f} | "
                f"latents_finite={torch.isfinite(latents).all().item()} | "
                f"prediction_finite={torch.isfinite(prediction).all().item()}"
            )

        if step % 25 == 0:
            print(f"Mean loss (last 25): {running_loss / 25:.6f}")
            running_loss = 0.0

# Save standalone LoRA only after successful training
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

state = get_peft_model_state_dict(pipe.unet)
state = convert_state_dict_to_diffusers(state)

StableDiffusionXLPipeline.save_lora_weights(
    save_directory=str(OUTPUT_DIR),
    unet_lora_layers=state,
    safe_serialization=True,
)

print("Training complete.")
print("Saved files:", [p.name for p in OUTPUT_DIR.iterdir()])

del optimizer, pipe
gc.collect()
torch.cuda.empty_cache()


GPU: Tesla T4
Images: 144


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [4]:

from pathlib import Path
from huggingface_hub import try_to_load_from_cache

model_id = "stabilityai/stable-diffusion-xl-base-1.0"

cache_root = Path("/root/.cache/huggingface/hub")
print("Cache exists:", cache_root.exists())
print("Cached model folders:")
if cache_root.exists():
    for p in cache_root.glob("models--stabilityai--stable-diffusion-xl-base-1.0"):
        print(p)

for filename in [
    "model_index.json",
    "unet/config.json",
    "vae/config.json",
    "text_encoder/config.json",
    "text_encoder_2/config.json",
]:
    result = try_to_load_from_cache(model_id, filename)
    print(filename, "FOUND" if isinstance(result, str) else "MISSING")


Cache exists: True
Cached model folders:
/root/.cache/huggingface/hub/models--stabilityai--stable-diffusion-xl-base-1.0
model_index.json FOUND
unet/config.json FOUND
vae/config.json FOUND
text_encoder/config.json FOUND
text_encoder_2/config.json FOUND


In [5]:
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    dtype=torch.float16,
    use_safetensors=True,
)
pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

StableDiffusionXLPipeline {
  "_class_name": "StableDiffusionXLPipeline",
  "_diffusers_version": "0.40.0",
  "_name_or_path": "stabilityai/stable-diffusion-xl-base-1.0",
  "feature_extractor": [
    null,
    null
  ],
  "force_zeros_for_empty_prompt": true,
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "EulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "text_encoder_2": [
    "transformers",
    "CLIPTextModelWithProjection"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    pipe.to("cuda")

    # VAEは数値安定性を優先してfloat32にする
    pipe.vae.to(dtype=torch.float32)

    print("GPU:", torch.cuda.get_device_name(0))
    print("UNet:", next(pipe.unet.parameters()).device)
    print("UNet dtype:", next(pipe.unet.parameters()).dtype)
    print("VAE dtype:", next(pipe.vae.parameters()).dtype)
    print(
        "VRAM allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


CUDA available: True
GPU: Tesla T4
UNet: cuda:0
UNet dtype: torch.float16
VAE dtype: torch.float32
VRAM allocated: 6.75 GB


In [8]:
print("UNet device:", next(pipe.unet.parameters()).device)
print("UNet dtype:", next(pipe.unet.parameters()).dtype)

print("LoRA parameters:")
for name, param in pipe.unet.named_parameters():
    if "lora_" in name:
        print(name, param.shape, param.requires_grad)
        break
else:
    print("LoRA not found")

UNet device: cuda:0
UNet dtype: torch.float16
LoRA parameters:
LoRA not found


In [9]:
import torch
from peft import LoraConfig, get_peft_model

# 前回のLoRAが残っていないことを確認してから実行
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.0,
    init_lora_weights="gaussian",
    target_modules=[
        "to_q",
        "to_k",
        "to_v",
        "to_out.0",
    ],
)

pipe.unet = get_peft_model(pipe.unet, lora_config)

# 学習可能パラメータを表示
pipe.unet.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [10]:
lora_params = [
    p for name, p in pipe.unet.named_parameters()
    if "lora_" in name and p.requires_grad
]

print("Trainable LoRA tensors:", len(lora_params))
print("Trainable parameters:", sum(p.numel() for p in lora_params))
print("Device:", lora_params[0].device if lora_params else "NONE")
print("Dtype:", lora_params[0].dtype if lora_params else "NONE")

assert len(lora_params) > 0, "LoRA parameters were not created"
assert all(torch.isfinite(p).all().item() for p in lora_params), \
    "LoRA contains non-finite values"

print("LoRA setup OK")

Trainable LoRA tensors: 0
Trainable parameters: 0
Device: NONE
Dtype: NONE


AssertionError: LoRA parameters were not created

In [11]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.0,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)

pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [2]:
import torch
import peft
import diffusers
import importlib.metadata as md

print("torch:", torch.__version__)
print("peft:", peft.__version__)
print("diffusers:", diffusers.__version__)

try:
    print("torchao:", md.version("torchao"))
except md.PackageNotFoundError:
    print("torchao: not installed")

print("pipe exists:", "pipe" in globals())

if "pipe" in globals():
    print("UNet type:", type(pipe.unet).__name__)
    print("UNet device:", next(pipe.unet.parameters()).device)

    lora_names = [
        name for name, _ in pipe.unet.named_parameters()
        if "lora_" in name
    ]
    print("LoRA parameter names:", len(lora_names))
    print("Sample module names:")
    for name, module in list(pipe.unet.named_modules())[:5]:
        print(" ", name, type(module).__name__)

torch: 2.11.0+cu128
peft: 0.20.0
diffusers: 0.40.0
torchao: 0.10.0
pipe exists: True
UNet type: UNet2DConditionModel
UNet device: cuda:0
LoRA parameter names: 0
Sample module names:
   UNet2DConditionModel
  conv_in Conv2d
  time_proj Timesteps
  time_embedding TimestepEmbedding
  time_embedding.linear_1 Linear


In [3]:
import inspect
from diffusers import StableDiffusionXLPipeline

print("add_adapter:", hasattr(pipe.unet, "add_adapter"))
print(
    "load_lora_adapter:",
    hasattr(pipe, "load_lora_adapter")
)
print(
    "LoraConfig available:",
    hasattr(__import__("peft"), "LoraConfig")
)
print("UNet class:", type(pipe.unet).__name__)


add_adapter: True
load_lora_adapter: False
LoraConfig available: True
UNet class: UNet2DConditionModel


In [4]:
import torch
from peft import LoraConfig

# UNetにLoRAを追加
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.0,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)

pipe.unet.add_adapter(lora_config)

# 学習対象を抽出
lora_params = [
    p for name, p in pipe.unet.named_parameters()
    if "lora_" in name and p.requires_grad
]

print("Trainable LoRA tensors:", len(lora_params))
print("Trainable parameters:", sum(p.numel() for p in lora_params))

assert len(lora_params) > 0, "LoRA parameters were not created"
assert all(torch.isfinite(p).all().item() for p in lora_params)

print("LoRA setup OK")

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported